In [17]:
import tensorflow as tf
from tensorflow.keras.layers import (
    TextVectorization,
    Embedding,
    Dense,
    LayerNormalization,
    MultiHeadAttention
)
from tensorflow.keras import Model

# -------------------------------
# Dataset
# -------------------------------

data = [
    ("i am a student", "je suis un etudiant"),
    ("how are you", "comment allez vous"),
    ("i love machine learning", "j aime apprentissage automatique"),
    ("good morning", "bonjour"),
    ("thank you", "merci"),
    ("see you later", "a plus tard"),
    ("what is your name", "quel est votre nom"),
    ("where are you going", "ou allez vous"),
    ("i like coffee", "j aime le cafe"),
    ("welcome", "bienvenue")
]

english_sentences = [x[0] for x in data]
french_sentences = ["start " + x[1] + " end" for x in data]

# -------------------------------
# Tokenization
# -------------------------------

vocab_size = 1000
sequence_length = 10

source_vectorization = TextVectorization(
    max_tokens=vocab_size,
    output_mode="int",
    output_sequence_length=sequence_length
)

target_vectorization = TextVectorization(
    max_tokens=vocab_size,
    output_mode="int",
    output_sequence_length=sequence_length
)

source_vectorization.adapt(english_sentences)
target_vectorization.adapt(french_sentences)

encoder_inputs_data = source_vectorization(english_sentences)
target_tokens = target_vectorization(french_sentences)

decoder_inputs_data = target_tokens[:, :-1]
decoder_targets = target_tokens[:, 1:]

# -------------------------------
# Positional Embedding
# -------------------------------

class PositionalEmbedding(tf.keras.layers.Layer):
    def __init__(self, sequence_length, vocab_size, embed_dim):
        super().__init__()

        self.token_embedding = Embedding(
            input_dim=vocab_size,
            output_dim=embed_dim
        )

        self.position_embedding = Embedding(
            input_dim=sequence_length,
            output_dim=embed_dim
        )

    def call(self, inputs):
        length = tf.shape(inputs)[-1]

        positions = tf.range(
            start=0,
            limit=length,
            delta=1
        )

        token_embeddings = self.token_embedding(inputs)
        position_embeddings = self.position_embedding(positions)

        return token_embeddings + position_embeddings

# -------------------------------
# Encoder
# -------------------------------

class TransformerEncoder(tf.keras.layers.Layer):

    def __init__(self, embed_dim, dense_dim, num_heads):
        super().__init__()

        self.attention = MultiHeadAttention(
            num_heads=num_heads,
            key_dim=embed_dim
        )

        self.dense_proj = tf.keras.Sequential([
            Dense(dense_dim, activation="relu"),
            Dense(embed_dim)
        ])

        self.layernorm1 = LayerNormalization()
        self.layernorm2 = LayerNormalization()

    def call(self, inputs):

        attention_output = self.attention(
            query=inputs,
            value=inputs,
            key=inputs
        )

        proj_input = self.layernorm1(
            inputs + attention_output
        )

        proj_output = self.dense_proj(
            proj_input
        )

        return self.layernorm2(
            proj_input + proj_output
        )

# -------------------------------
# Decoder
# -------------------------------

class TransformerDecoder(tf.keras.layers.Layer):

    def __init__(self, embed_dim, dense_dim, num_heads):
        super().__init__()

        self.self_attention = MultiHeadAttention(
            num_heads=num_heads,
            key_dim=embed_dim
        )

        self.cross_attention = MultiHeadAttention(
            num_heads=num_heads,
            key_dim=embed_dim
        )

        self.ffn = tf.keras.Sequential([
            Dense(dense_dim, activation="relu"),
            Dense(embed_dim)
        ])

        self.layernorm1 = LayerNormalization()
        self.layernorm2 = LayerNormalization()
        self.layernorm3 = LayerNormalization()

    def call(self, inputs, encoder_outputs):

        attention_output = self.self_attention(
            query=inputs,
            value=inputs,
            key=inputs,
            use_causal_mask=True
        )

        out1 = self.layernorm1(
            inputs + attention_output
        )

        attention_output2 = self.cross_attention(
            query=out1,
            value=encoder_outputs,
            key=encoder_outputs
        )

        out2 = self.layernorm2(
            out1 + attention_output2
        )

        ffn_output = self.ffn(out2)

        return self.layernorm3(
            out2 + ffn_output
        )

# -------------------------------
# Build Transformer
# -------------------------------

embed_dim = 128
dense_dim = 256
num_heads = 4

encoder_input = tf.keras.Input(
    shape=(None,),
    dtype="int64"
)

decoder_input = tf.keras.Input(
    shape=(None,),
    dtype="int64"
)

encoder_embeddings = PositionalEmbedding(
    sequence_length,
    vocab_size,
    embed_dim
)(encoder_input)

encoder_output = TransformerEncoder(
    embed_dim,
    dense_dim,
    num_heads
)(encoder_embeddings)

decoder_embeddings = PositionalEmbedding(
    sequence_length,
    vocab_size,
    embed_dim
)(decoder_input)

decoder_output = TransformerDecoder(
    embed_dim,
    dense_dim,
    num_heads
)(
    decoder_embeddings,
    encoder_output
)

final_output = Dense(
    vocab_size,
    activation="softmax"
)(decoder_output)

transformer = Model(
    [encoder_input, decoder_input],
    final_output
)

# -------------------------------
# Compile
# -------------------------------

transformer.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

transformer.summary()

# -------------------------------
# Train and Store History
# -------------------------------

history = transformer.fit(
    [encoder_inputs_data, decoder_inputs_data],
    decoder_targets,
    epochs=50,
    batch_size=2,
    verbose=1
)

# -------------------------------
# Final Accuracy and Loss
# -------------------------------

final_acc = history.history['accuracy'][-1]
final_loss = history.history['loss'][-1]

print("\nFinal Accuracy:", round(final_acc * 100, 2), "%")
print("Final Loss:", round(final_loss, 4))

Model: "functional_11"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_12      │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_13      │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ positional_embeddi… │ (None, None, 128) │    129,280 │ input_layer_12[0… │
│ (PositionalEmbeddi… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ positional_embeddi… │ (None, None, 128) │    129,280 │ input_layer_13[0… │
│ (PositionalEmbeddi… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ transformer_encode… │ (None, None, 128) │    330,240 │ positional_embed… │
│ (TransformerEncode… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ transformer_decode… │ (None, None, 128) │    594,304 │ positional_embed… │
│ (TransformerDecode… │                   │            │ transformer_enco… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_19 (Dense)    │ (None, None,      │    129,000 │ transformer_deco… │
│                     │ 1000)             │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 1,312,104 (5.01 MB)

 Trainable params: 1,312,104 (5.01 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/50
5/5 ━━━━━━━━━━━━━━━━━━━━ 4s 9ms/step - accuracy: 0.4667 - loss: 5.1379    
Epoch 2/50
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.5778 - loss: 3.3337 
Epoch 3/50
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.5778 - loss: 2.6111 
Epoch 4/50
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.6333 - loss: 1.9589 
Epoch 5/50
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.6889 - loss: 1.5882
Epoch 6/50
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.6889 - loss: 1.3239 
Epoch 7/50
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.7333 - loss: 1.0722 
Epoch 8/50
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.7889 - loss: 0.8717 
Epoch 9/50
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.8333 - loss: 0.7532 
Epoch 10/50
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.8556 - loss: 0.6200 
Epoch 11/50
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.9000 - loss: 0.5285 
Epoch 12/50
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.9000 - loss: 0.4578

In [18]:
sample = ["i love machine learning"]

enc_input = source_vectorization(sample)

prediction = transformer.predict(
    [enc_input, decoder_inputs_data[:1]]
)

predicted_ids = tf.argmax(prediction, axis=-1)

vocab = target_vectorization.get_vocabulary()

translated_words = []

for idx in predicted_ids.numpy()[0]:
    if idx < len(vocab):
        translated_words.append(vocab[idx])

print("Predicted Translation:")
print(" ".join(translated_words))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 235ms/step
Predicted Translation:
j suis un etudiant end    
